In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

In [2]:
bronx_matched = pd.read_csv('../data/processed/bronx_matched_clustered.csv')
bronx_matched['crash_date'] = pd.to_datetime(bronx_matched['crash_date'])

cluster_summary = pd.read_csv('../data/processed/bronx_cluster_summary.csv')

In [3]:
'cluster' in bronx_matched.columns

True

In [4]:
bronx_matched[['collision_id', 'cluster', 'number_of_persons_injured', 'number_of_persons_killed']].head()

,collision_id,cluster,number_of_persons_injured,number_of_persons_killed
0,72589,0,0.0,0.0
1,72590,1,0.0,0.0
2,72591,2,0.0,0.0
3,72593,3,1.0,0.0
4,72595,4,0.0,0.0


In [5]:
bronx_matched[['number_of_persons_injured', 'number_of_persons_killed']].sum()

number_of_persons_injured    79499.0
number_of_persons_killed       309.0
dtype: float64

In [6]:
severity_by_cluster = bronx_matched[bronx_matched['cluster'] != -1].groupby('cluster').agg(
    total_injured=('number_of_persons_injured', 'sum'),
    total_killed=('number_of_persons_killed', 'sum'),
    fatal_crash_count=('number_of_persons_killed', lambda x: (x > 0).sum()),
    injury_rate=('number_of_persons_injured', lambda x: (x > 0).mean() * 100)
).reset_index()

severity_by_cluster.head()

,cluster,total_injured,total_killed,fatal_crash_count,injury_rate
0,0,6.0,0.0,0,19.047619
1,1,9.0,0.0,0,35.714286
2,2,170.0,1.0,1,38.081395
3,3,43.0,0.0,0,34.615385
4,4,68.0,1.0,1,14.657980


In [7]:
cluster_summary = cluster_summary.merge(severity_by_cluster, on='cluster', how='left')
cluster_summary.head()

,cluster,crash_count,center_lat,center_lon,top_street,total_injured,total_killed,fatal_crash_count,injury_rate
0,0,21,40.821167,-73.907750,CAULDWELL AVENUE,6.0,0.0,0,19.047619
1,1,14,40.801668,-73.913235,EAST 134 STREET,9.0,0.0,0,35.714286
2,2,344,40.816082,-73.917685,EAST 149 STREET,170.0,1.0,1,38.081395
3,3,104,40.813233,-73.909077,EAST 149 STREET,43.0,0.0,0,34.615385
4,4,307,40.804697,-73.922393,BRUCKNER BOULEVARD,68.0,1.0,1,14.657980


In [8]:
cluster_summary.shape

(4465, 9)

In [9]:
scaler = MinMaxScaler()
cluster_summary[['crash_count_norm', 'injured_norm', 'killed_norm']] = scaler.fit_transform(
    cluster_summary[['crash_count', 'total_injured', 'total_killed']]
)

crash_weight = 1
injured_weight = 1
killed_weight = 2

cluster_summary['danger_score'] = (
    crash_weight * cluster_summary['crash_count_norm']
    + injured_weight * cluster_summary['injured_norm']
    + killed_weight * cluster_summary['killed_norm']
)

cluster_summary.sort_values('danger_score', ascending=False).head(10)

,cluster,crash_count,center_lat,center_lon,top_street,total_injured,total_killed,fatal_crash_count,injury_rate,crash_count_norm,injured_norm,killed_norm,danger_score
21,21,208,40.809109,-73.922889,EAST 138 STREET,106.0,4.0,4,37.980769,0.201754,0.226496,1.00,2.428250
317,317,989,40.820234,-73.890738,BRUCKNER BOULEVARD,373.0,1.0,1,25.884732,0.962963,0.797009,0.25,2.259972
3216,3216,1027,40.861863,-73.912780,WEST FORDHAM ROAD,468.0,0.0,0,29.016553,1.000000,1.000000,0.00,2.000000
1934,1934,531,40.878644,-73.871601,EAST GUN HILL ROAD,167.0,2.0,2,22.975518,0.516569,0.356838,0.50,1.873407
2530,2530,142,40.845985,-73.884472,SOUTHERN BOULEVARD,100.0,3.0,3,48.591549,0.137427,0.213675,0.75,1.851102
1153,1153,152,40.838727,-73.913770,GRAND CONCOURSE,77.0,3.0,2,36.842105,0.147173,0.164530,0.75,1.811703
1928,1928,663,40.878309,-73.870150,EAST GUN HILL ROAD,310.0,1.0,1,31.372549,0.645224,0.662393,0.25,1.807617
757,757,166,40.828419,-73.860677,WHITE PLAINS ROAD,56.0,3.0,3,23.493976,0.160819,0.119658,0.75,1.780477
93,93,370,40.818559,-73.927318,EAST 149 STREET,192.0,2.0,2,36.486486,0.359649,0.410256,0.50,1.769906
1809,1809,350,40.862754,-73.901083,JEROME AVENUE,167.0,2.0,1,37.714286,0.340156,0.356838,0.50,1.696994


In [10]:
bronx_matched[bronx_matched['cluster'] == 1153][['collision_id','crash_date','number_of_persons_injured','number_of_persons_killed']]

,collision_id,crash_date,number_of_persons_injured,number_of_persons_killed
9700,85246,2012-07-14,2.0,1.0
9747,85303,2012-07-21,0.0,0.0
9809,85380,2012-08-03,0.0,0.0
9843,85424,2012-08-13,1.0,0.0
9905,85496,2012-08-25,0.0,0.0
...,...,...,...,...
217042,4837677,2025-08-23,0.0,0.0
218108,4847065,2025-10-01,1.0,0.0
222113,4881527,2026-02-21,0.0,0.0
223688,4894445,2026-04-24,1.0,0.0


In [11]:
top_factor = bronx_matched[bronx_matched['cluster'] != -1].groupby('cluster')['contributing_factor_grouped'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
).reset_index(name='top_factor')

cluster_summary = cluster_summary.merge(top_factor, on='cluster', how='left')
cluster_summary

,cluster,crash_count,center_lat,center_lon,top_street,total_injured,total_killed,fatal_crash_count,injury_rate,crash_count_norm,injured_norm,killed_norm,danger_score,top_factor
0,0,21,40.821167,-73.907750,CAULDWELL AVENUE,6.0,0.0,0,19.047619,0.019493,0.012821,0.00,0.032314,Unspecified
1,1,14,40.801668,-73.913235,EAST 134 STREET,9.0,0.0,0,35.714286,0.012671,0.019231,0.00,0.031901,Other
2,2,344,40.816082,-73.917685,EAST 149 STREET,170.0,1.0,1,38.081395,0.334308,0.363248,0.25,1.197556,Unspecified
3,3,104,40.813233,-73.909077,EAST 149 STREET,43.0,0.0,0,34.615385,0.100390,0.091880,0.00,0.192270,Driver Inattention/Distraction
4,4,307,40.804697,-73.922393,BRUCKNER BOULEVARD,68.0,1.0,1,14.657980,0.298246,0.145299,0.25,0.943545,Unspecified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,4460,10,40.869189,-73.879750,NaN,4.0,0.0,0,30.000000,0.008772,0.008547,0.00,0.017319,Other
4461,4461,12,40.823417,-73.836520,NaN,4.0,0.0,0,25.000000,0.010721,0.008547,0.00,0.019268,Unspecified
4462,4462,10,40.855995,-73.844215,NaN,5.0,0.0,0,40.000000,0.008772,0.010684,0.00,0.019456,Unspecified
4463,4463,10,40.823474,-73.874699,NaN,6.0,0.0,0,40.000000,0.008772,0.012821,0.00,0.021592,Driver Inattention/Distraction


In [12]:
cluster_summary.to_csv('../data/processed/bronx_cluster_summary_scored.csv', index=False)